# Lesson 16 — Pivot Tables & Reshaping Data with Pandas

## Quality Engineering / Manufacturing Data Analysis

**Goal:** Learn how to convert raw production data into management-friendly summary tables using `pivot_table()`, `pivot()`, `melt()`, `stack()`, `unstack()`, and `crosstab()`.

### Learning outcomes
- Understand long vs wide data
- Create QA pivot tables
- Calculate yield and defect KPIs by Line, Shift, Model and Date
- Use multiple aggregation functions
- Add margins / grand totals
- Reshape data with `melt()` and `pivot()`
- Use `stack()` and `unstack()`
- Build cross-tabulations for defect analysis
- Solve practical manufacturing reporting problems


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
   

## 1. Create a Manufacturing Dataset

We will use inspection records similar to a real electronics manufacturing QA report.

In [2]:
df = pd.DataFrame({
    'Date': pd.to_datetime([
        '2026-09-01','2026-09-01','2026-09-01','2026-09-01',
        '2026-09-02','2026-09-02','2026-09-02','2026-09-02',
        '2026-09-03','2026-09-03','2026-09-03','2026-09-03',
        '2026-09-04','2026-09-04','2026-09-04','2026-09-04',
        '2026-09-05','2026-09-05','2026-09-05','2026-09-05'
    ]),
    'Line': ['L1','L1','L2','L2'] * 5,
    'Shift': ['A','B','A','B'] * 5,
    'Model': ['M1','M2','M1','M2'] * 5,
    'Inspected': [980,950,1020,990,1000,970,1010,980,1030,1000,1020,995,990,960,1040,1000,1015,985,1035,1010],
    'Defects': [18,25,12,30,15,28,20,24,17,31,14,26,22,29,13,27,16,24,19,32],
    'Cycle_Time_sec': [42,44,41,46,43,45,42,47,41,46,43,45,42,47,41,46,43,44,42,48]
})

df['Good'] = df['Inspected'] - df['Defects']
df['Defect_Rate'] = df['Defects'] / df['Inspected'] * 100
df['Yield'] = df['Good'] / df['Inspected'] * 100

df.head()

,Date,Line,Shift,Model,Inspected,Defects,Cycle_Time_sec,Good,Defect_Rate,Yield
0,2026-09-01,L1,A,M1,980,18,42,962,1.836735,98.163265
1,2026-09-01,L1,B,M2,950,25,44,925,2.631579,97.368421
2,2026-09-01,L2,A,M1,1020,12,41,1008,1.176471,98.823529
3,2026-09-01,L2,B,M2,990,30,46,960,3.030303,96.969697
4,2026-09-02,L1,A,M1,1000,15,43,985,1.500000,98.500000


## 2. Long Data vs Wide Data

**Long format:** one observation per row. This is usually convenient for analysis.

**Wide format:** categories become columns. This is often convenient for reports and dashboards.

Pivoting helps us move between these representations.

## 3. Basic `pivot_table()`

In [3]:
# Average defect rate by Line and Shift
pd.pivot_table(
    df,
    index='Line',
    columns='Shift',
    values='Defect_Rate',
    aggfunc='mean'
).round(2)

Shift,A,B
Line,,
L1,1.76,2.82
L2,1.52,2.79


## 4. Pivot Table for Production Quantity

In [4]:
pd.pivot_table(
    df,
    index='Line',
    columns='Shift',
    values='Inspected',
    aggfunc='sum'
)

Shift,A,B
Line,,
L1,5015,4865
L2,5125,4975


## 5. Multiple Values in One Pivot Table

A pivot table can summarize several measures simultaneously.

In [5]:
pivot_quality = pd.pivot_table(
    df,
    index='Line',
    columns='Shift',
    values=['Inspected', 'Defects', 'Yield'],
    aggfunc={'Inspected':'sum', 'Defects':'sum', 'Yield':'mean'}
)

pivot_quality.round(2)

Defects      Inspected        Yield       
Shift       A    B         A     B      A      B
Line                                            
L1         88  137      5015  4865  98.24  97.18
L2         78  139      5125  4975  98.48  97.21

## 6. Pivot by Line and Model

In [6]:
pd.pivot_table(
    df,
    index='Line',
    columns='Model',
    values='Defects',
    aggfunc='sum'
)

Model,M1,M2
Line,,
L1,88,137
L2,78,139


## 7. Grand Total with `margins=True`

This is useful for management reports, but remember that rates should be interpreted carefully.

In [7]:
pd.pivot_table(
    df,
    index='Line',
    columns='Shift',
    values='Defects',
    aggfunc='sum',
    margins=True,
    margins_name='Grand Total'
)

Shift,A,B,Grand Total
Line,,,
L1,88,137,225
L2,78,139,217
Grand Total,166,276,442


## 8. Multiple Aggregation Functions

In [8]:
pd.pivot_table(
    df,
    index='Line',
    values='Cycle_Time_sec',
    aggfunc=['mean', 'min', 'max', 'std']
).round(2)

,mean,min,max,std
,Cycle_Time_sec,Cycle_Time_sec,Cycle_Time_sec,Cycle_Time_sec
Line,,,,
L1,43.7,41,47,1.89
L2,44.1,41,48,2.60


## 9. `fill_value` for Missing Combinations

In [9]:
pd.pivot_table(
    df,
    index='Line',
    columns='Model',
    values='Defects',
    aggfunc='sum',
    fill_value=0
)

Model,M1,M2
Line,,
L1,88,137
L2,78,139


## 10. Three-Dimensional QA Pivot

Use multiple indexes and columns when you need a more detailed report.

In [10]:
three_dimensional = pd.pivot_table(
    df,
    index=['Line', 'Model'],
    columns='Shift',
    values='Defects',
    aggfunc='sum',
    fill_value=0
)

three_dimensional

Shift        A    B
Line Model         
L1   M1     88    0
     M2      0  137
L2   M1     78    0
     M2      0  139

## 11. `pivot()` vs `pivot_table()`

`pivot()` expects each index-column combination to be unique. `pivot_table()` can aggregate duplicate combinations.

In [11]:
small = df[['Date', 'Line', 'Defects']].copy()

# Each Date-Line combination is unique in this dataset.
wide = small.pivot(index='Date', columns='Line', values='Defects')
wide

ValueError: Index contains duplicate entries, cannot reshape

## 12. `melt()` — Wide to Long

In [ ]:
wide_df = pd.DataFrame({
    'Line': ['L1', 'L2', 'L3'],
    'M1_Defects': [18, 12, 21],
    'M2_Defects': [25, 30, 24],
    'M3_Defects': [14, 19, 17]
})

wide_df

In [ ]:
long_df = wide_df.melt(
    id_vars='Line',
    var_name='Model',
    value_name='Defects'
)

long_df

The `melt()` result is much easier to group, filter and visualize.

## 13. Clean the melted Model column

In [ ]:
long_df['Model'] = long_df['Model'].str.replace('_Defects', '', regex=False)
long_df

## 14. `stack()` and `unstack()`

In [ ]:
grouped = df.groupby(['Line', 'Shift'])['Defects'].sum()
grouped

In [ ]:
# MultiIndex Series -> table
grouped.unstack('Shift')

In [ ]:
# Table -> stacked Series
grouped.unstack('Shift').stack()

## 15. `crosstab()` for QA Counts

In [ ]:
# Create a simple quality status category
df['Quality_Status'] = np.where(df['Defect_Rate'] <= 2.0, 'Within Target', 'Above Target')

pd.crosstab(df['Line'], df['Quality_Status'])

## 16. Crosstab with percentages

In [ ]:
pd.crosstab(
    df['Line'],
    df['Quality_Status'],
    normalize='index'
) * 100

## 17. Manufacturing QA Report

Create a compact report showing total inspected, total defects and average yield by Line and Shift.

In [ ]:
report = pd.pivot_table(
    df,
    index='Line',
    columns='Shift',
    values=['Inspected', 'Defects', 'Yield'],
    aggfunc={'Inspected':'sum', 'Defects':'sum', 'Yield':'mean'},
    fill_value=0
)

report.round(2)

## 18. Important QA Calculation: Weighted Defect Rate

Do not calculate an overall defect rate by simply averaging percentages when sample sizes differ.

Correct approach:

**Overall Defect Rate = Total Defects / Total Inspected × 100**

In [ ]:
overall_inspected = df['Inspected'].sum()
overall_defects = df['Defects'].sum()
overall_defect_rate = overall_defects / overall_inspected * 100

overall_defect_rate

## 19. Practice Exercises

Try these yourself before checking the hints:

1. Create a pivot table with `Line` as index, `Model` as columns and total `Inspected` as values.
2. Create a pivot table showing average `Cycle_Time_sec` by Line and Shift.
3. Create a pivot table showing total Defects by Date and Line.
4. Add grand totals to the Date-Line defect table.
5. Convert `wide_df` back to wide format using `pivot()`.
6. Use `crosstab()` to count Line vs Shift combinations.
7. Create a table showing the percentage of records above the defect-rate target for each shift.
8. Create a management-ready table with Line as rows and Shift as columns for Yield.


In [ ]:
# Hints
# 1: pd.pivot_table(df, index='Line', columns='Model', values='Inspected', aggfunc='sum')
# 2: pd.pivot_table(df, index='Line', columns='Shift', values='Cycle_Time_sec', aggfunc='mean')
# 3: pd.pivot_table(df, index='Date', columns='Line', values='Defects', aggfunc='sum')
# 4: Use margins=True
# 5: long_df.pivot(index='Line', columns='Model', values='Defects')
# 6: pd.crosstab(df['Line'], df['Shift'])
# 7: pd.crosstab(df['Shift'], df['Quality_Status'], normalize='index') * 100
# 8: pd.pivot_table(df, index='Line', columns='Shift', values='Yield', aggfunc='mean')

# Mini Project — Production Quality Report

Build a final report with:

- Rows: `Line`
- Columns: `Shift`
- Values: total Inspected, total Defects, average Yield and average Cycle Time
- Grand Total
- Missing combinations filled with zero where appropriate

Then reshape the result into a long format suitable for Power BI or a database table.

### QA interpretation questions
1. Which Line-Shift combination has the highest defect count?
2. Which combination has the lowest average yield?
3. Are conclusions based on counts or rates?
4. Could a high defect count simply be caused by higher production volume?
5. What additional information would you need before starting RCA?

# Key Takeaways

- `pivot_table()` = powerful aggregation + reshaping tool.
- `pivot()` = reshape when combinations are unique.
- `melt()` = wide → long.
- `stack()` = columns → index level.
- `unstack()` = index level → columns.
- `crosstab()` = frequency/count tables.
- For quality rates, always consider the denominator.

**Next lesson:** We will move from data summarization toward **data cleaning and missing-value handling**, which is essential before any reliable statistical or ML analysis.